---
title: Cross-Sectional Momentum — a worked example
summary: A 12-1 month momentum strategy, demonstrated on the ConvexPi synthetic market with an out-of-sample check.
tags: [momentum, equities, template]
---

# Cross-Sectional Momentum — a worked example

This is a **template post**. Tell a clear story, show runnable code and charts, and define your
strategy as a class named `MyStrategy` that subclasses the ConvexPi `Strategy` — then it can be
**scored on the permanent out-of-sample leaderboard** (we re-grade it on hidden data; your own
numbers below are for the narrative).

In [ ]:
# Setup: the ConvexPi Lab SDK (installed automatically in Colab).
try:
    import convexpi.lab  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "convexpi-lab"])
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
print("ready")

## The idea

Stocks that outperformed over the past ~12 months tend to keep outperforming next month. We rank the
cross-section by the `mom_12m` feature (12-month return, skipping the most recent month), go long the
top quintile and short the bottom, rebalancing as the market evolves.

In [ ]:
# --- MyStrategy: keep this self-contained in one cell so it can be graded ---
import numpy as np
from convexpi.lab import Strategy

class MyStrategy(Strategy):
    """12-1 cross-sectional momentum: long the top-quintile mom_12m, short the bottom."""
    quintile_frac = 0.20

    def on_day(self, day, features, prices, portfolio):
        sig = np.nan_to_num(features.get("mom_12m", np.zeros(len(prices))))
        n = len(sig)
        k = max(1, int(n * self.quintile_frac))
        order = np.argsort(sig)
        w = np.zeros(n)
        w[order[-k:]] = 1.0 / k     # long winners
        w[order[:k]]  = -1.0 / k    # short losers
        return w

## In-sample vs out-of-sample

The `Grader` trains on the first half of a synthetic market and evaluates on the held-out second
half — the same out-of-sample discipline used for the leaderboard.

In [ ]:
from convexpi.lab import SyntheticMarket, Grader

market = SyntheticMarket(n_stocks=60, n_days=1500, seed=0)
report = Grader(market).evaluate(MyStrategy())

print(f"in-sample Sharpe   : {report.is_sharpe:+.2f}")
print(f"out-of-sample Sharpe: {report.oos_sharpe:+.2f}")
print(f"overfitting ratio  : {report.overfitting_ratio:+.2f}  (1.0 = holds up; <0 = noise)")

oos = report.oos_result.daily_returns
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.cumprod(1 + oos))
ax.set_title("Out-of-sample equity curve"); ax.set_ylabel("growth of $1")
plt.tight_layout(); plt.show()

## What I'd try next

- Combine `mom_12m` with `qual_roe` or `val_bm` to diversify the signal.
- Skew by `vol_1m` to lighten up on the most volatile names.
- Watch the **overfitting ratio**: a beautiful in-sample Sharpe that collapses out of sample is the
  lesson, not the goal.

*Open this in Colab, edit `MyStrategy`, and submit it to the leaderboard from your published post.*